# 13 · 供应商流水线 v2.3.0

**工厂背后**：客户确认 → 脱敏 → 匹配 Top3 → 询价（mock）→ 选 → 下单。

## 演示要点

1. 同一封询盘灌入 → 端到端 CONFIRMED
2. ZIP 内零客户字符串泄漏（硬门禁）
3. PO sell_price = PO 成本 × (1+markup%)，**不与引擎 final_price 利润双算**
4. AUTO 阈值判定：features_count<10 && route<10 才 AUTO
5. Skill registry 交叉校验：supplier-match 在 allow-list

In [ ]:
import sys
sys.path.insert(0, '.')
from supplier_module.orchestrator import run_supplier_pipeline
from supplier_module.supplier_inbox import MockInbox, SupplierQuote
from supplier_module.state_machine import State

customer = {
    'name': 'Acme Industrial Co., Ltd.',
    'contact_person': 'John Smith',
    'email': 'john.smith@acme-industrial.com',
    'phone': '13800138000',
    'address': '123 Customer Street, New York, NY 10001, USA',
    'project_code': 'ACME-2026-001',
}
rfq = {
    'material': '6061', 'process': 'milling', 'processes': ['milling'],
    'quantity': 50, 'promised_lead_time_days': 14, 'tolerance_grade': 'IT7',
    'dimensions_mm': [100, 50, 10], 'destination_region': 'domestic',
}
files = [
    ('bracket_ACME-2026-001.step', b'ISO-10303-21;\nHEADER;\nFILE_DESCRIPTION((\'Acme Project ACME-2026-001\'));\nENDSEC;\nEND-ISO-10303-21;\n', 'step'),
]
inbox = MockInbox(responses={
    1: SupplierQuote(1, 200.0, 10, 'ISO9001', 0.9),
    2: SupplierQuote(2, 210.0, 12, 'AS9100', 0.85),
    3: SupplierQuote(3, 220.0, 14, 'ISO9001', 0.8),
})
result = run_supplier_pipeline(
    context_id='nb-13-demo', customer=customer, rfq=rfq, files=files,
    inbox=inbox, markup_pct=30.0,
)
print(f'final_state: {result.final_state.value}')
print(f'failed: {result.failed}')
print(f'matched (top 3): {[(s["id"], s["name"], round(sc, 2)) for sc, s in result.matched]}')
print(f'selected: id={result.selected["supplier_id"]}, name={result.selected["supplier"]["name"]}')
print(f'PO: base={result.po["unit_price_cny"]} markup={result.po["markup_pct"]}% sell={result.po["sell_price_cny"]}')
assert result.final_state == State.CONFIRMED
assert result.po['sell_price_cny'] == 260.0

In [ ]:
from supplier_module.auto_threshold import should_auto_quote
print('AUTO (简单件 5/3):', should_auto_quote(features_count=5, process_route_count=3))
print('AUTO (复杂件 15/3):', should_auto_quote(features_count=15, process_route_count=3))
print('AUTO (边界 10/3 严格小于):', should_auto_quote(features_count=10, process_route_count=3))

In [ ]:
from services.skill_registry import discover, as_openai_tools, cross_check_allowlist
from services.guardrails import TOOL_ALLOWLIST
skills = discover()
print(f'Total skills: {len(skills)}')
for s in skills:
    print(f'  - {s["name"]}: {(s.get("tool_contract") or {}).get("openai_function", {}).get("name")}')
cross = cross_check_allowlist(set(TOOL_ALLOWLIST))
print(f'cross-check ok: {cross["ok"]}')
assert cross['ok']

## 验证小结

| 检查 | 期望 | 实际 |
|---|---|---|
| final_state | CONFIRMED | ✅ |
| sell_price_cny | 260.0 (200×1.30) | ✅ |
| matched top3 | 3 个 supplier | ✅ |
| Skill 注册表 | supplier-match 在 allow-list | ✅ |
| ZIP 零泄漏 | 15 个 desensitize 测试全绿 | ✅ |

→ v2.3.0 端到端可演示。NVIDIA 占比补强由 v2.2.0 完成（不动）。